# 9. TeleopXR + Franka + MuJoCo（采集版）

本 notebook 目标：
1. 在 MuJoCo 中使用 Franka Panda 模型。
2. IK 使用 TeleopXR 库（FrankaRobot + PyrokiSolver + IKController）。
3. 食指 Trigger 控制夹爪开合。
4. 支持 RECORD=True/False 两种模式。


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
TELEOPXR_ROOT = Path(r"C:\tbxr_real")

print('PROJECT_ROOT =', PROJECT_ROOT)
print('TELEOPXR_ROOT =', TELEOPXR_ROOT)

assert (PROJECT_ROOT / 'mujoco_env').exists(), 'Open this notebook from lerobot-mujoco-tutorial root.'
assert TELEOPXR_ROOT.exists(), 'C:\tbxr_real not found. Check TeleopXR source path.'

# Put local teleop_xr source path first in sys.path
if str(TELEOPXR_ROOT) not in sys.path:
    sys.path.insert(0, str(TELEOPXR_ROOT))


PROJECT_ROOT = C:\Users\kewei\lerobot-mujoco-tutorial
TELEOPXR_ROOT = C:\tbxr_real


## 1) 安装与依赖

如果环境已安装，可跳过。


In [2]:
# %pip install -U pip setuptools wheel
# %pip install -e C:/tbxr_real
# %pip install robot_descriptions


## 2) 初始化 TeleopXR IK 与 MuJoCo Franka

规则：双手 Grip/Squeeze 同时按住才进入控制，双击触发重置。


In [3]:
import time
import threading
import numpy as np
import mujoco
import mujoco.viewer

from teleop_xr import Teleop
from teleop_xr.config import TeleopSettings
from teleop_xr.messages import XRState, XRDeviceRole, XRHandedness
from teleop_xr.events import EventProcessor, EventSettings, XRButton

from teleop_xr.ik.robots.franka import FrankaRobot
from teleop_xr.ik.solver import PyrokiSolver
from teleop_xr.ik.controller import IKController

from robot_descriptions import panda_mj_description

# ---------- TeleopXR server ----------
TELEOP_HOST = '0.0.0.0'
TELEOP_PORT = 4444

# ---------- Runtime ----------
CTRL_HZ = 30
RUN_SECONDS = 180
TRIGGER_THRESHOLD = 0.6

state_lock = threading.Lock()
shared = {
    'latest_state': None,          # XRState
    'reset_requested': False,
}

def on_xr_state(_pose, info_dict):
    try:
        state = XRState.model_validate(info_dict)
    except Exception:
        return
    with state_lock:
        shared['latest_state'] = state

# Double squeeze event triggers reset
processor = EventProcessor(EventSettings(double_press_threshold_ms=350, long_press_threshold_ms=500))

def on_reset(_event):
    with state_lock:
        shared['reset_requested'] = True

processor.on_double_press(button=XRButton.SQUEEZE, callback=on_reset)

teleop = Teleop(TeleopSettings(host=TELEOP_HOST, port=TELEOP_PORT, input_mode='controller'))
teleop.subscribe(on_xr_state)
teleop.subscribe(processor.process)

server_thread = threading.Thread(target=teleop.run, daemon=True)
server_thread.start()

# ---------- IK ----------
robot = FrankaRobot()
solver = PyrokiSolver(robot)
controller = IKController(robot, solver)
q_target = np.array(robot.get_default_config(), dtype=np.float32)

# ---------- MuJoCo model ----------
model = mujoco.MjModel.from_xml_path(panda_mj_description.MJCF_PATH)
data = mujoco.MjData(model)

print('TeleopXR URL:', f'https://<LAN_IP>:{TELEOP_PORT}')
print('MuJoCo model loaded from:', panda_mj_description.MJCF_PATH)
print('IK actuated joints:', robot.actuated_joint_names)


{"host":"0.0.0.0","port":4444,"robot_vis":null,"input_mode":"controller","speed":1.0,"natural_phone_orientation_euler":[0.0,-0.7853981633974483,0.0],"natural_phone_position":[0.0,0.0,0.0],"camera_views":{},"video_config":null}
Server started at 0.0.0.0:4444
The phone web app should be available at https://192.168.209.36:4444
2026-03-01 16:30:25.572 | INFO     | pyroki._robot_urdf_parser:_topologically_sort_joints:199 - Joints were not in topological order; they will be internally sorted.
2026-03-01 16:30:26.658 | INFO     | jaxls._py310._problem:analyze:121 - Building optimization problem with 4 terms and 1 variables: 4 costs, 0 eq_zero, 0 leq_zero, 0 geq_zero
2026-03-01 16:30:29.444 | INFO     | jaxls._py310._problem:analyze:229 - Vectorizing group with 1 costs, 1 variables each: manipulability_residual
2026-03-01 16:30:29.704 | INFO     | jaxls._py310._problem:analyze:229 - Vectorizing group with 1 costs, 1 variables each: rest_residual
2026-03-01 16:30:29.866 | INFO     | jaxls._py3

TeleopXR URL: https://<LAN_IP>:4444
MuJoCo model loaded from: C:\Users\kewei\.cache\robot_descriptions\mujoco_menagerie\franka_emika_panda\panda.xml
IK actuated joints: ['panda_joint1', 'panda_joint2', 'panda_joint3', 'panda_joint4', 'panda_joint5', 'panda_joint6', 'panda_joint7']


In [4]:
import math
import teleop_xr
from teleop_xr.messages import XRDeviceRole, XRHandedness

# A. 放宽 pose jump 判定（减少频繁 reset）
_orig_are_close = teleop_xr.are_close
def _are_close_relaxed(a, b=None, lin_tol=1e-9, ang_tol=1e-9):
    # 线位移 20cm、角度 80° 以内不判定为 jump
    return _orig_are_close(a, b, lin_tol=0.20, ang_tol=math.radians(80))
teleop_xr.are_close = _are_close_relaxed

# B. deadman 改为“仅右手 Squeeze(Grip)”
def _deadman_right_only(self, state):
    for d in state.devices:
        if d.role == XRDeviceRole.CONTROLLER and d.handedness == XRHandedness.RIGHT:
            if d.gamepad is None:
                return False
            return (len(d.gamepad.buttons) > 1 and bool(d.gamepad.buttons[1].pressed))
    return False

controller._check_deadman = _deadman_right_only.__get__(controller, type(controller))
print("patched: deadman=RIGHT squeeze only, pose jump threshold relaxed")


patched: deadman=RIGHT squeeze only, pose jump threshold relaxed


## 3) 关节映射与辅助函数


In [5]:
def mat_to_rpy(R):
    # XYZ extrinsic (roll-pitch-yaw)
    sy = np.sqrt(R[0, 0] * R[0, 0] + R[1, 0] * R[1, 0])
    singular = sy < 1e-6
    if not singular:
        roll = np.arctan2(R[2, 1], R[2, 2])
        pitch = np.arctan2(-R[2, 0], sy)
        yaw = np.arctan2(R[1, 0], R[0, 0])
    else:
        roll = np.arctan2(-R[1, 2], R[1, 1])
        pitch = np.arctan2(-R[2, 0], sy)
        yaw = 0.0
    return np.array([roll, pitch, yaw], dtype=np.float32)


def get_trigger_value(xr_state):
    if xr_state is None:
        return 0.0
    for d in xr_state.devices:
        if d.role == XRDeviceRole.CONTROLLER and d.handedness == XRHandedness.RIGHT and d.gamepad is not None:
            if len(d.gamepad.buttons) > 0:
                return float(d.gamepad.buttons[0].value)
    return 0.0


def map_joint_qpos_indices(model, target_joint_names):
    # 建立 model 名称表
    model_joint_names = [mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_JOINT, i) for i in range(model.njnt)]
    model_joint_set = set(model_joint_names)

    idx = []
    valid_names = []

    for name in target_joint_names:
        candidates = [name]

        # 兼容 panda_joint1 <-> joint1
        if name.startswith("panda_"):
            candidates.append(name.replace("panda_", "", 1))
        else:
            candidates.append("panda_" + name)

        # 再做一次通配：按 joint数字后缀匹配
        if "joint" in name:
            suffix = name[name.find("joint"):]  # e.g. joint1
            candidates.append(suffix)

        hit = None
        for c in candidates:
            if c in model_joint_set:
                hit = c
                break

        if hit is not None:
            jid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, hit)
            idx.append(int(model.jnt_qposadr[jid]))
            valid_names.append(hit)

    return valid_names, idx


# arm: 用 IK 输出关节名作为输入，内部自动做别名匹配
ik_arm_joint_names = list(robot.actuated_joint_names[:7])
arm_joint_names, arm_qpos_idx = map_joint_qpos_indices(model, ik_arm_joint_names)

# gripper
gripper_candidates = ["panda_finger_joint1", "panda_finger_joint2", "finger_joint1", "finger_joint2"]
gripper_names, gripper_qpos_idx = map_joint_qpos_indices(model, gripper_candidates)

print("IK arm joints:", ik_arm_joint_names)
print("Mapped arm joints:", list(zip(arm_joint_names, arm_qpos_idx)))
print("Mapped gripper joints:", list(zip(gripper_names, gripper_qpos_idx)))

assert len(arm_qpos_idx) == 7, f"Arm joint mapping failed, got {len(arm_qpos_idx)}"




def apply_targets_to_mujoco(data, q_arm, gripper_open_ratio):
    # arm
    n = min(len(q_arm), len(arm_qpos_idx))
    for i in range(n):
        data.qpos[arm_qpos_idx[i]] = q_arm[i]

    # gripper: open_ratio=1 open, 0 close
    for jname, qidx in zip(gripper_names, gripper_qpos_idx):
        jid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, jname)
        if jid >= 0:
            qmin, qmax = model.jnt_range[jid]
            val = qmin + float(gripper_open_ratio) * (qmax - qmin)
            data.qpos[qidx] = val

    mujoco.mj_forward(model, data)


IK arm joints: ['panda_joint1', 'panda_joint2', 'panda_joint3', 'panda_joint4', 'panda_joint5', 'panda_joint6', 'panda_joint7']
Mapped arm joints: [('joint1', 0), ('joint2', 1), ('joint3', 2), ('joint4', 3), ('joint5', 4), ('joint6', 5), ('joint7', 6)]
Mapped gripper joints: [('finger_joint1', 7), ('finger_joint2', 8), ('finger_joint1', 7), ('finger_joint2', 8)]


## 4) 采集参数（record 开关）


In [6]:
import os
from PIL import Image
from lerobot.common.datasets.lerobot_dataset import LeRobotDataset

RECORD = True  # True: record dataset, False: debug only

REPO_NAME = 'datawhale_eai_franka_xr'
ROOT = './demo_data_franka_xr'
ROBOT_TYPE = 'franka'
TASK_NAME = 'XR teleoperation with Franka in MuJoCo'
NUM_DEMO = 1

# Control and sampling parameters
SIM_MODE = 'kinematic'  # 'kinematic' = XR-like responsiveness, 'physics' = dynamic realism
SIM_FPS = 90
DATASET_FPS = 30
CAPTURE_EVERY_N = max(1, SIM_FPS // DATASET_FPS)
FPS = SIM_FPS  # keep compatibility with downstream cells

dataset = None
record_flag = False
episode_id = 0

if RECORD:
    create_new = True
    if os.path.exists(ROOT):
        print(f'Directory {ROOT} already exists.')
        ans = input('Delete and recreate it (y/n) ')
        if ans.strip().lower() == 'y':
            import shutil
            shutil.rmtree(ROOT)
        else:
            create_new = False

    if create_new:
        dataset = LeRobotDataset.create(
            repo_id=REPO_NAME,
            root=ROOT,
            robot_type=ROBOT_TYPE,
            fps=DATASET_FPS,
            features={
                'observation.image': {
                    'dtype': 'image',
                    'shape': (256, 256, 3),
                    'names': ['height', 'width', 'channels'],
                },
                'observation.wrist_image': {
                    'dtype': 'image',
                    'shape': (256, 256, 3),
                    'names': ['height', 'width', 'channels'],
                },
                'observation.state': {
                    'dtype': 'float32',
                    'shape': (6,),
                    'names': ['state'],
                },
                'action': {
                    'dtype': 'float32',
                    'shape': (8,),  # 7 arm + 1 gripper
                    'names': ['action'],
                },
                'obj_init': {
                    'dtype': 'float32',
                    'shape': (6,),
                    'names': ['obj_init'],
                },
            },
            image_writer_threads=10,
            image_writer_processes=5,
        )
    else:
        dataset = LeRobotDataset(REPO_NAME, root=ROOT)

print(f'SIM_MODE={SIM_MODE}, SIM_FPS={SIM_FPS}, DATASET_FPS={DATASET_FPS}, CAPTURE_EVERY_N={CAPTURE_EVERY_N}')


Directory ./demo_data_franka_xr already exists.


Delete and recreate it (y/n)  y


SIM_MODE=kinematic, SIM_FPS=90, DATASET_FPS=30, CAPTURE_EVERY_N=3


## 5) 主循环（IK 跟随 + 可选采集）

- 右手 Trigger 控制夹爪。
- RECORD=True 时写入训练数据。


In [8]:
renderer = mujoco.Renderer(model, 256, 256)

obs_buffer = []
act_buffer = []

start_t = time.time()
step_count = 0

with mujoco.viewer.launch_passive(model, data) as viewer:
    while viewer.is_running():
        tic = time.time()

        now = time.time()
        if (not RECORD) and (now - start_t > RUN_SECONDS):
            print('Debug mode timeout reached.')
            break

        with state_lock:
            xr_state = shared['latest_state']
            reset_requested = bool(shared['reset_requested'])
            shared['reset_requested'] = False

        # reset event
        if reset_requested:
            controller.reset()
            q_target = np.array(robot.get_default_config(), dtype=np.float32)
            apply_targets_to_mujoco(data, q_target[:7], gripper_open_ratio=1.0)

            if RECORD and record_flag and dataset is not None:
                dataset.save_episode()
                episode_id += 1
                print(f'Episode saved. episode_id={episode_id}')
                record_flag = False

                if episode_id >= NUM_DEMO:
                    print('Target number of demos reached.')
                    break

        # IK step (deadman rule is inside IKController)
        if xr_state is not None:
            q_target = np.array(controller.step(xr_state, q_target), dtype=np.float32)

        deadman_active = bool(controller.active)

        # trigger -> gripper_open_ratio
        trig = get_trigger_value(xr_state)
        gripper_open_ratio = float(np.clip(1.0 - trig, 0.0, 1.0))

        # apply to mujoco
        apply_targets_to_mujoco(data, q_target[:7], gripper_open_ratio)

        if SIM_MODE == 'kinematic':
            # Kinematic direct-drive: closest behavior to XR hand tracking
            data.qvel[:] = 0.0
            if hasattr(data, 'qacc'):
                data.qacc[:] = 0.0
            mujoco.mj_forward(model, data)
        else:
            # Physics mode: more realistic, less responsive
            mujoco.mj_step(model, data)

        # render image
        renderer.update_scene(data)
        img = renderer.render()
        wrist_img = img.copy()  # Use same image if wrist camera is not available

        # ee state (x,y,z,r,p,y)
        ee_body_name = 'panda_hand'
        bid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, ee_body_name)
        if bid < 0:
            bid = model.nbody - 1
        p = data.xpos[bid].copy()
        R = data.xmat[bid].reshape(3, 3).copy()
        rpy = mat_to_rpy(R)
        ee_state = np.concatenate([p, rpy], dtype=np.float32)

        action_vec = np.concatenate([q_target[:7], np.array([1.0 - gripper_open_ratio], dtype=np.float32)], dtype=np.float32)

        if RECORD:
            if deadman_active and (not record_flag):
                record_flag = True
                print(f'Start recording episode {episode_id + 1}')

            # Physics mode: more realistic, less responsive
            if record_flag and dataset is not None and (step_count % CAPTURE_EVERY_N == 0):
                dataset.add_frame(
                    {
                        'observation.image': np.asarray(Image.fromarray(img).resize((256, 256))),
                        'observation.wrist_image': np.asarray(Image.fromarray(wrist_img).resize((256, 256))),
                        'observation.state': ee_state,
                        'action': action_vec,
                        'obj_init': np.zeros(6, dtype=np.float32),
                    },
                    task=TASK_NAME,
                )
        else:
            obs_buffer.append(ee_state.copy())
            act_buffer.append(action_vec.copy())

        viewer.sync()

        step_count += 1
        if step_count % FPS == 0:
            print(f'steps={step_count}, mode={SIM_MODE}, deadman={deadman_active}, trigger={trig:.2f}, gripper={1.0-gripper_open_ratio:.2f}, record={record_flag}')

        # Control loop pacing
        elapsed = time.time() - tic
        time.sleep(max(0.0, 1.0 / FPS - elapsed))

        if step_count % FPS == 0 and xr_state is not None:
            for d in xr_state.devices:
                if d.role == XRDeviceRole.CONTROLLER:
                    pressed = []
                    if d.gamepad is not None:
                        for i, b in enumerate(d.gamepad.buttons):
                            if b.pressed:
                                pressed.append(i)
                    print(f"{d.handedness}: pressed={pressed}")
            print(f"controller.active={controller.active}")


print('Loop finished.')
if RECORD:
    print(f'RECORD mode done. Episodes saved = {episode_id}')
else:
    print('DEBUG mode done.', 'frames =', len(obs_buffer), 'actions =', len(act_buffer))


steps=90, mode=kinematic, deadman=False, trigger=0.00, gripper=0.00, record=False
left: pressed=[]
right: pressed=[]
controller.active=False
steps=180, mode=kinematic, deadman=False, trigger=0.00, gripper=0.00, record=False
left: pressed=[]
right: pressed=[]
controller.active=False
steps=270, mode=kinematic, deadman=False, trigger=0.00, gripper=0.00, record=False
left: pressed=[]
right: pressed=[]
controller.active=False
steps=360, mode=kinematic, deadman=False, trigger=0.00, gripper=0.00, record=False
left: pressed=[]
right: pressed=[]
controller.active=False
steps=450, mode=kinematic, deadman=False, trigger=0.00, gripper=0.00, record=False
left: pressed=[]
right: pressed=[]
controller.active=False
steps=540, mode=kinematic, deadman=False, trigger=0.00, gripper=0.00, record=False
left: pressed=[]
right: pressed=[]
controller.active=False
steps=630, mode=kinematic, deadman=False, trigger=0.00, gripper=0.00, record=False
left: pressed=[]
right: pressed=[]
controller.active=False
steps=7

Pose jump detected, resetting the pose
Pose jump detected, resetting the pose


steps=1260, mode=kinematic, deadman=False, trigger=0.00, gripper=0.00, record=False
controller.active=False


Pose jump detected, resetting the pose
Pose jump detected, resetting the pose


steps=1350, mode=kinematic, deadman=False, trigger=0.00, gripper=0.00, record=False
left: pressed=[]
right: pressed=[]
controller.active=False
steps=1440, mode=kinematic, deadman=False, trigger=0.00, gripper=0.00, record=False
left: pressed=[]
right: pressed=[]
controller.active=False
steps=1530, mode=kinematic, deadman=False, trigger=0.00, gripper=0.00, record=False
left: pressed=[]
right: pressed=[]
controller.active=False


...

2026-03-01 16:37:55.446 | DEBUG    | teleop_xr.ik.controller:step:242 - [IKController] New Config: [-0.02807726 -1.0085369  -0.06870136 -2.2888641  -0.48846093  1.4383134
  1.1546973 ]
2026-03-01 16:37:55.467 | DEBUG    | teleop_xr.ik.controller:step:242 - [IKController] New Config: [-0.03187118 -1.01204    -0.07507763 -2.2985384  -0.49311775  1.4357996
  1.1524589 ]
2026-03-01 16:37:55.491 | DEBUG    | teleop_xr.ik.controller:step:242 - [IKController] New Config: [-0.03320434 -1.014362   -0.07972267 -2.3049562  -0.49373743  1.4327055
  1.1496629 ]
2026-03-01 16:37:55.513 | DEBUG    | teleop_xr.ik.controller:step:242 - [IKController] New Config: [-0.03469432 -1.0166237  -0.08366839 -2.311645   -0.49587968  1.4294355
  1.1462955 ]
2026-03-01 16:37:55.539 | DEBUG    | teleop_xr.ik.controller:step:242 - [IKController] New Config: [-0.03578988 -1.0181577  -0.08542716 -2.3155005  -0.49766105  1.4287314
  1.1454023 ]
2026-03-01 16:37:55.568 | DEBUG    | teleop_xr.ik.controller:step:242 - [IK

Map:   0%|          | 0/771 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/8 [00:00<?, ?ba/s]

Episode saved. episode_id=2
Target number of demos reached.
Loop finished.
RECORD mode done. Episodes saved = 2


Pose jump detected, resetting the pose
Pose jump detected, resetting the pose
Pose jump detected, resetting the pose
Pose jump detected, resetting the pose


## 6) 调试轨迹保存（仅 RECORD=False 生效）


In [ ]:
from pathlib import Path
import numpy as np

save_dir = Path('./demo_data_franka_xr_debug')
save_dir.mkdir(parents=True, exist_ok=True)
save_path = save_dir / 'franka_xr_trace.npz'

if not RECORD and len(obs_buffer) > 0:
    np.savez_compressed(
        save_path,
        obs=np.asarray(obs_buffer, dtype=np.float32),
        action=np.asarray(act_buffer, dtype=np.float32),
    )
    print('saved debug trace to', save_path.resolve())
else:
    print('No debug trace to save (or RECORD=True).')

if RECORD:
    print('Dataset root =', Path(ROOT).resolve())
    print('Repo name =', REPO_NAME)


## 7) 常见问题

1. 端口被占用：换端口或先 kill 进程。
2. 没有 XR 数据：确认头显网络与防火墙。
3. 夹爪不动：检查 trigger 变化和 gripper 映射。
